----
<div style="display: flex; align-items: center;">
    <img src="https://developers.lseg.com/content/dam/devportal/icons/logo/lseg-logo.svg" width="20%" style="vertical-align: top;">
    <h1 style="margin-left: 20px;">Data Library for Python</h1>
</div>

----

## Content layer - Custom Instruments - Streaming
This standalone notebook demonstrates a custom instrument snapshot, streaming callbacks and the stream's local data cache with `lseg.data.content.custom_instruments.Definition`.

**Prerequisites:** a configured LSEG Data session with access to Custom Instruments, the streaming service and underlying market data. Creating the example requires `trapi.synthetic.crud` for platform sessions. See [Manage and Search](EX-2.12.01-CustomInstruments-ManageAndSearch.ipynb) for instrument management and current basket/UDC limitations.

The notebook creates one unique instrument on the service. Streams are closed explicitly or with a context manager, and the final cleanup cell deletes only this instrument. Run cleanup before closing the session, including if an intermediate cell fails. Delete the previous instrument before re-running the creation cell.

#### Learn more

Visit the [LSEG Developer Portal](https://developers.lseg.com/en) for learning materials about the LSEG Data Library for Python.

#### Getting Help and Support

For questions about the API, visit the [LSEG Developer Community](https://community.developers.lseg.com/).

----

## Set the configuration file location
Run the notebook from its own directory. Session settings and credentials are read from `lseg-data.config.json` in the project's `Configuration` directory.

In [ ]:
import os
os.environ["LD_LIB_CONFIG_PATH"] = "../../../Configuration"

## Some Imports to start with

In [ ]:
import asyncio
from uuid import uuid4

import lseg.data as ld
from lseg.data.content import custom_instruments as ci

## Open the data session
`open_session()` selects the configured default session. Use `ld.open_session("platform.rdp")` instead to select that configured platform session explicitly.

In [ ]:
ld.open_session()

## Create a formula instrument
Use the full returned symbol to identify the instrument in the streaming requests.

In [ ]:
instrument = ci.manage.create_formula(
    symbol=f"Live{uuid4().hex[:8]}",
    formula="GBP=*3",
    instrument_name="GBP Triple",
    currency="USD",
    time_zone="LON",
    description="Custom Instruments streaming example",
)
symbol = instrument.symbol
symbol

## Retrieve a snapshot without subscribing to updates
`Definition(...).get_stream()` builds a stream. `open(with_updates=False)` requests the initial data without ongoing updates; `get_snapshot()` returns the cached values as a DataFrame. The `finally` block closes the stream even if opening it or reading the cache raises an exception. Status and error messages are displayed rather than discarded.

Calculated fields may arrive in a second refresh after the initial refresh and completion notification. The short asynchronous pause allows those messages to be processed without blocking the notebook event loop; it does not guarantee that every field is populated. Availability depends on the service, entitlements and underlying data.

In [ ]:
snapshot_stream = ci.Definition(
    universe=symbol,
    fields=["TRDPRC_1", "TRADE_DATE", "SALTIM_NS"],
).get_stream()
snapshot_stream.on_status(lambda status, name, stream: print("Status", name, status))
snapshot_stream.on_error(lambda error, name, stream: print("Error", name, error))

try:
    snapshot_stream.open(with_updates=False)
    await asyncio.sleep(3)
    display(snapshot_stream.get_snapshot())
finally:
    snapshot_stream.close()

## Register streaming callbacks
`on_refresh` receives the initial image and any later refresh images. `on_update` receives incremental changes. `on_status` and `on_error` report service status and errors. These callbacks receive `(data, instrument_name, stream)`; `on_complete` receives only `(stream)`.

Completion signals the initial stream response, not the end of the live subscription. Further refreshes can arrive afterwards. Keep callbacks short: they run as messages are processed.

In [ ]:
def on_refresh(data, name, stream):
    print("Refresh", name, data)


def on_update(data, name, stream):
    print("Update", name, data)


def on_status(status, name, stream):
    print("Status", name, status)


def on_error(error, name, stream):
    print("Error", name, error)


def on_complete(stream):
    print("Initial response complete")

## Open the stream and inspect its cache
The asynchronous context manager opens the stream with updates and closes it on exit, including when the cell raises an exception. Jupyter supports top-level `await`. `asyncio.sleep` lets the event loop process messages during the observation period.

No update is guaranteed during this period, especially outside market hours. The tests reliably observe refresh and completion, but do not assume an update count. `get_snapshot()` reads the latest local cache rather than making a separate historical-data request.

For a platform session whose configuration requires explicit endpoint selection, the tests also use `ci.Definition(universe=symbol, api="streaming.custom-instruments.endpoints.resource")`.

In [ ]:
stream = ci.Definition(
    universe=symbol,
    fields=["TRDPRC_1", "TRADE_DATE", "SALTIM_NS"],
).get_stream()
stream.on_refresh(on_refresh)
stream.on_update(on_update)
stream.on_status(on_status)
stream.on_error(on_error)
stream.on_complete(on_complete)

async with stream:
    await asyncio.sleep(10)
    display(stream.get_snapshot())
    display(stream.get_snapshot(universe=symbol, fields=["TRDPRC_1"]))

### Dynamic subscriptions
The stream also exposes `add_fields`, `remove_fields`, `add_instruments` and `remove_instruments`. Adding or removing a subscription is not the same as creating or deleting a stored custom instrument. The corresponding tests are currently marked as expected failures because update events are unstable; this introductory notebook does not rely on dynamic subscription changes.

## Delete the example instrument
Both streams have been closed before deletion. If an earlier request fails, ensure the streams are closed and run this cell before closing the session. Only the instrument created above is deleted.

In [ ]:
ci.manage.delete(universe=symbol)

## Close the session

In [ ]:
ld.close_session()